In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType, DoubleType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("catalogo", "catalog_tr")
dbutils.widgets.text("esquema", "bronze")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

source_table = f"{catalogo}.raw.`googledrive-source`"

In [0]:
drive_schema = StructType(fields=[
    StructField("transaction_id", StringType(), False),
    StructField("transaction_date", TimestampType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("store_location", StringType(), True),
    StructField("payment_method", StringType(), True)
])

df_drive = (
    spark.read.table(source_table)
    .drop("_gdrive_metadata", "_file_metadata")
    .select(
        col("transaction_id"),
        to_timestamp(col("transaction_date")).alias("transaction_date"),
        col("customer_id"),
        col("product_id"),
        col("product_name"),
        col("category"),
        col("quantity").cast(IntegerType()),
        col("unit_price").cast(DoubleType()),
        col("total_amount").cast(DoubleType()),
        col("store_location"),
        col("payment_method")
    )
)

In [0]:
(
    df_drive.write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalogo}.{esquema}.drive_bronze")
)